In [ ]:
!pip install langchain chromadb google-genai tiktoken pypdf langchain_google_genai langchain_community


In [ ]:
!pip install langchain-google-genai google-generativeai

In [16]:
!pip install python-dotenv langchain langchain-core langchain-google-genai google-generativeai langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.3 MB/s eta 0:00:00


In [4]:
!pip install -qU langchain-chroma

In [5]:
from langchain_chroma import Chroma

In [11]:
!pip install langchain-community

In [14]:
from langchain_core.documents import Document

doc1 = Document(
    page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and remarkable consistency, he has been a key player for Royal Challengers Bangalore.",
    metadata={"team": "Royal Challengers Bangalore"}
)

doc2 = Document(
    page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to multiple titles. He is known for his elegant batting and calm leadership.",
    metadata={"team": "Mumbai Indians"}
)

doc3 = Document(
    page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to several IPL championships. His finishing abilities and wicketkeeping skills make him one of the greatest players in IPL history.",
    metadata={"team": "Chennai Super Kings"}
)

doc4 = Document(
    page_content="Jasprit Bumrah is regarded as one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his deadly yorkers and exceptional death-over bowling.",
    metadata={"team": "Mumbai Indians"}
)

doc5 = Document(
    page_content="Ravindra Jadeja is a world-class all-rounder who has been a vital part of Chennai Super Kings. His contributions with bat, ball, and in the field make him a complete cricketer.",
    metadata={"team": "Chennai Super Kings"}
)




In [26]:
# creating a list in which uploading all the docs created above
docs = [doc1, doc2, doc3, doc4, doc5]

In [31]:
import os
os.environ['GOOGLE_API_KEY'] = 'AQ.Ab8RN6KrSnigBhyDDJ90Agu9ZakVZFRJVXJH_dDcz_7tciXztA'

In [32]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [33]:
vector_store = Chroma(
    embedding_function= GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview", output_dimensionality = 32),
    persist_directory='chroma-db',
    collection_name = 'sample'

)

In [34]:
# add document
vector_store.add_documents(docs)

['0c005868-2315-4f32-910f-9bd38c0e2e38',
 '17d329a7-a4b6-4133-bdb7-03a3c3c1b55b',
 'c2f09a11-50a8-4cab-adca-a9b70d099de0',
 'fa06ac99-a90a-404f-8869-91dc1e8463ad',
 '9775ca38-8bd7-4c4b-bb3b-d7b7f67d4556']

In [36]:
# view documents
vector_store.get(include = ['documents', 'metadatas'])

{'ids': ['0c005868-2315-4f32-910f-9bd38c0e2e38',
  '17d329a7-a4b6-4133-bdb7-03a3c3c1b55b',
  'c2f09a11-50a8-4cab-adca-a9b70d099de0',
  'fa06ac99-a90a-404f-8869-91dc1e8463ad',
  '9775ca38-8bd7-4c4b-bb3b-d7b7f67d4556'],
 'embeddings': None,
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and remarkable consistency, he has been a key player for Royal Challengers Bangalore.',
  'Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to multiple titles. He is known for his elegant batting and calm leadership.',
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to several IPL championships. His finishing abilities and wicketkeeping skills make him one of the greatest players in IPL history.',
  'Jasprit Bumrah is regarded as one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his deadly yorkers and exceptional death-ov

In [38]:
# similarity search
vector_store.similarity_search(
    query = "who among these is the bowler?",
    k = 1
)

[Document(id='fa06ac99-a90a-404f-8869-91dc1e8463ad', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is regarded as one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his deadly yorkers and exceptional death-over bowling.')]

In [46]:
# search with similarity score
results = vector_store.similarity_search_with_score(
    query = "Whos is the bowler among them?",
    k = 5
)

for docs,score in results:
  print(f"Score: {score}")
  print(f"Content: {docs.page_content[:50]}")
  print("---")


Score: 0.2063756287097931
Content: Jasprit Bumrah is regarded as one of the best fast
---
Score: 0.27212584018707275
Content: Virat Kohli is one of the most successful and cons
---
Score: 0.2927766442298889
Content: Ravindra Jadeja is a world-class all-rounder who h
---
Score: 0.3217979073524475
Content: Rohit Sharma is the most successful captain in IPL
---
Score: 0.33331000804901123
Content: MS Dhoni, famously known as Captain Cool, has led 
---


In [49]:
# metadata filtering OPTION 1
vector_store.similarity_search_with_score(
    query = " ",
    filter = {'team': 'Chennai Super Kings'}
)

[(Document(id='9775ca38-8bd7-4c4b-bb3b-d7b7f67d4556', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a world-class all-rounder who has been a vital part of Chennai Super Kings. His contributions with bat, ball, and in the field make him a complete cricketer.'),
  0.37058326601982117),
 (Document(id='c2f09a11-50a8-4cab-adca-a9b70d099de0', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to several IPL championships. His finishing abilities and wicketkeeping skills make him one of the greatest players in IPL history.'),
  0.3909521698951721)]

In [50]:
# metadata filtering OPTION 2
#USE GET METHOD

vector_store.get(
    where = {'team': 'Chennai Super Kings'}
)

{'ids': ['c2f09a11-50a8-4cab-adca-a9b70d099de0',
  '9775ca38-8bd7-4c4b-bb3b-d7b7f67d4556'],
 'embeddings': None,
 'documents': ['MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to several IPL championships. His finishing abilities and wicketkeeping skills make him one of the greatest players in IPL history.',
  'Ravindra Jadeja is a world-class all-rounder who has been a vital part of Chennai Super Kings. His contributions with bat, ball, and in the field make him a complete cricketer.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'team': 'Chennai Super Kings'},
  {'team': 'Chennai Super Kings'}]}

In [51]:
#update document
update_doc1 = Document(
    page_content = "Somethinf about virat kohli",
    metadata = {"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id = "0c005868-2315-4f32-910f-9bd38c0e2e38", document=update_doc1)

In [56]:
#view document
vector_store.get(include = ['documents', 'metadatas'])

{'ids': ['0c005868-2315-4f32-910f-9bd38c0e2e38',
  '17d329a7-a4b6-4133-bdb7-03a3c3c1b55b',
  'c2f09a11-50a8-4cab-adca-a9b70d099de0',
  'fa06ac99-a90a-404f-8869-91dc1e8463ad',
  '9775ca38-8bd7-4c4b-bb3b-d7b7f67d4556'],
 'embeddings': None,
 'documents': ['Somethinf about virat kohli',
  'Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to multiple titles. He is known for his elegant batting and calm leadership.',
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to several IPL championships. His finishing abilities and wicketkeeping skills make him one of the greatest players in IPL history.',
  'Jasprit Bumrah is regarded as one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his deadly yorkers and exceptional death-over bowling.',
  'Ravindra Jadeja is a world-class all-rounder who has been a vital part of Chennai Super Kings. His contributions with bat, ball, and in the field make him a comp

In [57]:
#delete document
vector_store.delete(ids = ["0c005868-2315-4f32-910f-9bd38c0e2e38"])

In [59]:
vector_store.get(include = ['documents', 'metadatas'])

{'ids': ['17d329a7-a4b6-4133-bdb7-03a3c3c1b55b',
  'c2f09a11-50a8-4cab-adca-a9b70d099de0',
  'fa06ac99-a90a-404f-8869-91dc1e8463ad',
  '9775ca38-8bd7-4c4b-bb3b-d7b7f67d4556'],
 'embeddings': None,
 'documents': ['Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to multiple titles. He is known for his elegant batting and calm leadership.',
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to several IPL championships. His finishing abilities and wicketkeeping skills make him one of the greatest players in IPL history.',
  'Jasprit Bumrah is regarded as one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his deadly yorkers and exceptional death-over bowling.',
  'Ravindra Jadeja is a world-class all-rounder who has been a vital part of Chennai Super Kings. His contributions with bat, ball, and in the field make him a complete cricketer.'],
 'uris': None,
 'included': ['documents', 'metadatas'],
